In [ ]:
# # Roberta ABSA
#
# Multi-head model za detekciju višestrukih kategorija i klasifikaciju sentimenta po kategorijama.
#
# ACD:
# 11 nezavisnih kategorija
# BCEWithLogitsLoss
#
# ACSA:
# 4 sentimenta po kategoriji
# CrossEntropyLoss
#
# Najbolji model:
# najveći macro-f1 na validacionom skupu
#
# Pragovi:
# podešeni na validacionom skupu nakon treniranja
#
# Test:
# evaluira se jednom korišćenjem najboljeg zamrznutog modela i podešenih pragova

In [1]:
pip install -q torch transformers scikit-learn tqdm sentencepiece iterative-stratification

In [2]:
import json
import copy
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    precision_recall_fscore_support
)

from tqdm.auto import tqdm


In [5]:
# Configuration
def get_config():
    return {
        "input_file": "annotations.json",

        "model_name": "xlm-roberta-base",

        "model_folder": "weights",

        "model_name_prefix": "roberta_absa",

        "experiment_name": "runs/roberta_absa",

        "max_length": 512,

        "seed": 42,

        "val_size": 0.10,

        "test_size": 0.10,

        "num_epochs": 20,

        "batch_size": 16,

        "learning_rate": 2e-5,

        "weight_decay": 0.01,

        "warmup_ratio": 0.1,

        "dropout": 0.2,

        "gradient_clip": 1.0,

        "early_stopping_patience": 10,

        "num_workers": 0,
    }


config = get_config()

CATEGORIES = [
    "Baterija",
    "Kamera",
    "Ekran",
    "Memorija",
    "Zvučnici",
    "Izgled",
    "Hardver",
    "Softver",
    "Cena",
    "Performanse",
    "Opšta ocena",
]

SENTIMENTS = [
    "Pozitivan",
    "Negativan",
    "Neutralan",
    "Konflikt",
]

CATEGORY_TO_ID = {
    category: i
    for i, category in enumerate(CATEGORIES)
}

SENTIMENT_TO_ID = {
    sentiment: i
    for i, sentiment in enumerate(SENTIMENTS)
}

ID_TO_CATEGORY = {
    i: category
    for category, i in CATEGORY_TO_ID.items()
}

ID_TO_SENTIMENT = {
    i: sentiment
    for sentiment, i in SENTIMENT_TO_ID.items()
}

NUM_CATEGORIES = len(CATEGORIES)
NUM_SENTIMENTS = len(SENTIMENTS)


# %%
random.seed(config["seed"])
np.random.seed(config["seed"])
torch.manual_seed(config["seed"])

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        config["seed"]
    )

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )




Device: cuda
GPU: Tesla T4


In [7]:
# ## Load and preprocess data

def load_examples(path):

    with open(
        path,
        "r",
        encoding="utf-8-sig"
    ) as f:
        raw_data = json.load(f)

    print(
        "Loaded:",
        path,
        len(raw_data)
    )

    examples = []

    for item in raw_data:

        example = preprocess_item(item)

        if example is not None:
            examples.append(example)

    print(
        "Usable:",
        len(examples)
    )

    return examples


def preprocess_item(item):

    text = item.get(
        "comment",
        ""
    ).strip()

    if not text:
        return None

    category_presence = np.zeros(
        NUM_CATEGORIES,
        dtype=np.float32
    )

    sentiment_labels = np.full(
        NUM_CATEGORIES,
        -1,
        dtype=np.int64
    )

    for aspect in item.get(
        "aspect_categories",
        []
    ):

        category = aspect.get(
            "category"
        )

        polarity = aspect.get(
            "polarity"
        )

        if (
            category not in CATEGORY_TO_ID
            or
            polarity not in SENTIMENT_TO_ID
        ):
            continue

        category_id = CATEGORY_TO_ID[
            category
        ]

        category_presence[
            category_id
        ] = 1.0

        sentiment_labels[
            category_id
        ] = SENTIMENT_TO_ID[
            polarity
        ]

    if np.sum(category_presence) == 0:
        return None

    return {
        "text": text,
        "category_presence": category_presence,
        "sentiment_labels": sentiment_labels,
    }


# Load the already-stratified splits

train_examples = load_examples(
    "train.json"
)

val_examples = load_examples(
    "validation.json"
)

test_examples = load_examples(
    "test.json"
)

Loaded: train.json 14041
Usable: 5174
Loaded: validation.json 1755
Usable: 628
Loaded: test.json 1755
Usable: 631


In [8]:
def make_stratification_labels(examples):

    labels = []

    for example in examples:

        y = np.zeros(
            NUM_CATEGORIES * NUM_SENTIMENTS,
            dtype=int
        )

        for category_id in range(NUM_CATEGORIES):

            if example["category_presence"][category_id] == 1:

                sentiment_id = (
                    example["sentiment_labels"][category_id]
                )

                if sentiment_id >= 0:

                    label_id = (
                        category_id * NUM_SENTIMENTS
                        + sentiment_id
                    )

                    y[label_id] = 1

        labels.append(y)

    return np.array(labels)

In [9]:
# ## Train / validation / test split


train_labels = make_stratification_labels(train_examples)
val_labels = make_stratification_labels(val_examples)
test_labels = make_stratification_labels(test_examples)

stratification_labels = np.concatenate(
    [
        train_labels,
        val_labels,
        test_labels
    ],
    axis=0
)



print(
    "Train:",
    len(train_examples)
)

print(
    "Validation:",
    len(val_examples)
)

print(
    "Test:",
    len(test_examples)
)

Train: 5174
Validation: 628
Test: 631


In [10]:
def print_distribution(name, labels):

    counts = labels.sum(axis=0)
    percentages = counts / len(labels) * 100

    print(f"\n{name}:")

    for category_id, category in ID_TO_CATEGORY.items():

        for sentiment_id, sentiment in ID_TO_SENTIMENT.items():

            label_id = (
                category_id * NUM_SENTIMENTS
                + sentiment_id
            )

            print(
                f"{category:15s} + "
                f"{sentiment:12s}: "
                f"{counts[label_id]:4.0f} "
                f"({percentages[label_id]:6.2f}%)"
            )


print_distribution(
    "Full dataset",
    stratification_labels
)

print_distribution(
    "Train",
    train_labels
)

print_distribution(
    "Validation",
    val_labels
)

print_distribution(
    "Test",
    test_labels
)


Full dataset:
Baterija        + Pozitivan   : 1037 ( 16.12%)
Baterija        + Negativan   :  942 ( 14.64%)
Baterija        + Neutralan   :  125 (  1.94%)
Baterija        + Konflikt    :   95 (  1.48%)
Kamera          + Pozitivan   :  763 ( 11.86%)
Kamera          + Negativan   :  489 (  7.60%)
Kamera          + Neutralan   :   71 (  1.10%)
Kamera          + Konflikt    :  115 (  1.79%)
Ekran           + Pozitivan   :  537 (  8.35%)
Ekran           + Negativan   :  376 (  5.84%)
Ekran           + Neutralan   :   34 (  0.53%)
Ekran           + Konflikt    :   21 (  0.33%)
Memorija        + Pozitivan   :   71 (  1.10%)
Memorija        + Negativan   :   87 (  1.35%)
Memorija        + Neutralan   :    5 (  0.08%)
Memorija        + Konflikt    :    1 (  0.02%)
Zvučnici        + Pozitivan   :  215 (  3.34%)
Zvučnici        + Negativan   :  256 (  3.98%)
Zvučnici        + Neutralan   :    6 (  0.09%)
Zvučnici        + Konflikt    :   28 (  0.44%)
Izgled          + Pozitivan   :  561 (  8.72%

In [11]:

# ## Tokenizer

tokenizer = AutoTokenizer.from_pretrained(
    config["model_name"]
)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [12]:
# Dataset

class ABSADataset(Dataset):

    def __init__(
        self,
        examples,
        tokenizer,
        max_length
    ):

        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(
            self.examples
        )

    def __getitem__(self, index):

        example = self.examples[index]

        encoding = self.tokenizer(
            example["text"],
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )

        return {
            "input_ids": torch.tensor(
                encoding["input_ids"],
                dtype=torch.long
            ),

            "attention_mask": torch.tensor(
                encoding["attention_mask"],
                dtype=torch.long
            ),

            "category_presence": torch.tensor(
                example["category_presence"],
                dtype=torch.float32
            ),

            "sentiment_labels": torch.tensor(
                example["sentiment_labels"],
                dtype=torch.long
            ),

            "text": example["text"],
        }


def collate_batch(batch):

    padded = tokenizer.pad(
        {
            "input_ids": [
                x["input_ids"]
                for x in batch
            ],

            "attention_mask": [
                x["attention_mask"]
                for x in batch
            ],
        },
        padding=True,
        return_tensors="pt",
    )

    return {
        "input_ids":
            padded["input_ids"],

        "attention_mask":
            padded["attention_mask"],

        "category_presence":
            torch.stack([
                x["category_presence"]
                for x in batch
            ]),

        "sentiment_labels":
            torch.stack([
                x["sentiment_labels"]
                for x in batch
            ]),

        "text": [
            x["text"]
            for x in batch
        ],
    }


train_dataset = ABSADataset(
    train_examples,
    tokenizer,
    config["max_length"]
)

val_dataset = ABSADataset(
    val_examples,
    tokenizer,
    config["max_length"]
)

test_dataset = ABSADataset(
    test_examples,
    tokenizer,
    config["max_length"]
)


train_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=True,
    num_workers=config["num_workers"],
    collate_fn=collate_batch,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=config["num_workers"],
    collate_fn=collate_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=config["num_workers"],
    collate_fn=collate_batch,
)


print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Test batches:",
    len(test_loader)
)


# ## Loss weights

train_presence = np.array([
    x["category_presence"]
    for x in train_examples
])

positive_counts = (
    train_presence.sum(axis=0)
)

negative_counts = (
    len(train_examples)
    -
    positive_counts
)

category_pos_weight = np.ones(
    NUM_CATEGORIES,
    dtype=np.float32
)

for i in range(
    NUM_CATEGORIES
):

    if positive_counts[i] > 0:

        category_pos_weight[i] = (
            negative_counts[i]
            /
            positive_counts[i]
        )


print(
    "Category positive weights:"
)

for i, category in (
    ID_TO_CATEGORY.items()
):

    print(
        f"{category:15s}: "
        f"{category_pos_weight[i]:.3f}"
    )


sentiment_counts = np.zeros(
    NUM_SENTIMENTS,
    dtype=np.float32
)

for example in train_examples:

    for label in (
        example["sentiment_labels"]
    ):

        if label >= 0:
            sentiment_counts[
                label
            ] += 1


sentiment_weights = np.ones(
    NUM_SENTIMENTS,
    dtype=np.float32
)

total_sentiments = (
    sentiment_counts.sum()
)

for i in range(NUM_SENTIMENTS):

    if sentiment_counts[i] > 0:

        sentiment_weights[i] = np.sqrt(
            total_sentiments
            /
            (
                NUM_SENTIMENTS
                *
                sentiment_counts[i]
            )
        )

print(
    "\nSentiment weights:"
)

for i, sentiment in (
    ID_TO_SENTIMENT.items()
):

    print(
        f"{sentiment:12s}: "
        f"{sentiment_weights[i]:.3f}"
    )





Train batches: 324
Validation batches: 40
Test batches: 40
Category positive weights:
Baterija       : 1.940
Kamera         : 3.499
Ekran          : 5.676
Memorija       : 38.197
Zvučnici       : 11.807
Izgled         : 6.677
Hardver        : 3.649
Softver        : 2.496
Cena           : 6.531
Performanse    : 4.736
Opšta ocena    : 1.086

Sentiment weights:
Pozitivan   : 0.683
Negativan   : 0.793
Neutralan   : 2.732
Konflikt    : 2.754


In [13]:
# BERTić model

class BERTicABSA(nn.Module):

    def __init__(
        self,
        model_name,
        num_categories,
        num_sentiments,
        dropout
    ):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            model_name
        )

        hidden_size = (
            self.encoder.config.hidden_size
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.category_head = nn.Linear(
            hidden_size,
            num_categories
        )

        self.category_embedding = nn.Embedding(
            num_categories,
            64
        )

        self.sentiment_head = nn.Linear(
            hidden_size + 64,
            num_sentiments
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled = outputs.last_hidden_state[:, 0]

        pooled = self.dropout(pooled)

        # Category prediction
        category_logits = self.category_head(
            pooled
        )

        # Create category embeddings
        batch_size = pooled.size(0)

        category_ids = torch.arange(
            NUM_CATEGORIES,
            device=pooled.device
        )

        category_emb = self.category_embedding(
            category_ids
        )

        # [batch, 1, hidden]
        pooled = pooled.unsqueeze(1)

        # [batch, 11, hidden]
        pooled = pooled.expand(
            -1,
            NUM_CATEGORIES,
            -1
        )

        # [1, 11, 64] → [batch, 11, 64]
        category_emb = category_emb.unsqueeze(0).expand(
            batch_size,
            -1,
            -1
        )

        # [batch, 11, hidden + 64]
        sentiment_input = torch.cat(
            [pooled, category_emb],
            dim=-1
        )

        # [batch, 11, 4]
        sentiment_logits = self.sentiment_head(
            sentiment_input
        )

        return {
            "category_logits": category_logits,
            "sentiment_logits": sentiment_logits
        }


In [14]:

model = BERTicABSA(
    config["model_name"],
    NUM_CATEGORIES,
    NUM_SENTIMENTS,
    config["dropout"]
).to(DEVICE)

USE_AMP = (
    DEVICE.type == "cuda"
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

print(
    f"Parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)




# ## Loss functions


category_loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        category_pos_weight,
        dtype=torch.float32,
        device=DEVICE
    )
)

sentiment_loss_fn = nn.CrossEntropyLoss(
    weight=torch.tensor(
        sentiment_weights,
        dtype=torch.float32,
        device=DEVICE
    ),
    ignore_index=-1
)




model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parameters: 278,056,143


In [15]:
# Joint loss

def calculate_loss(
    outputs,
    category_targets,
    sentiment_targets
):

    category_loss = category_loss_fn(
        outputs["category_logits"],
        category_targets
    )

    sentiment_logits = (
        outputs["sentiment_logits"]
    )

    valid = (
        sentiment_targets >= 0
    )

    if valid.any():

        sentiment_loss = F.cross_entropy(
            sentiment_logits[valid],
            sentiment_targets[valid],
            weight=sentiment_loss_fn.weight,
        )

    else:

        sentiment_loss = torch.tensor(
            0.0,
            device=DEVICE
        )

    total_loss = (
        1.0 * category_loss
        +
        1.0 * sentiment_loss
    )

    return (
        total_loss,
        category_loss,
        sentiment_loss
    )


# ## Optimizer

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"]
)



total_training_steps = (
    len(train_loader)
    *
    config["num_epochs"]
)

warmup_steps = int(
    total_training_steps
    *
    config["warmup_ratio"]
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)




In [16]:
# ## Train one epoch

def train_one_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    writer,
    global_step
):

    model.train()

    total_loss = 0.0
    total_category_loss = 0.0
    total_sentiment_loss = 0.0

    total_examples = 0

    progress = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for batch in progress:

        input_ids = batch[
            "input_ids"
        ].to(DEVICE)

        attention_mask = batch[
            "attention_mask"
        ].to(DEVICE)

        category_targets = batch[
            "category_presence"
        ].to(DEVICE)

        sentiment_targets = batch[
            "sentiment_labels"
        ].to(DEVICE)

        optimizer.zero_grad(
            set_to_none=True
        )

        #AMP
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            (
                loss,
                category_loss,
                sentiment_loss
            ) = calculate_loss(
                outputs,
                category_targets,
                sentiment_targets
            )

        scaler.scale(
            loss
        ).backward()

        # Unscale before gradient clipping
        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            config["gradient_clip"]
        )

        old_scale = scaler.get_scale()

        scaler.step(
            optimizer
        )

        scaler.update()

        new_scale = scaler.get_scale()

        #if new_scale < old_scale:
        #    print(
        #        "AMP: optimizer step skipped "
        #        "due to Inf/NaN gradients"
        #    )

        scheduler.step()

        # Statistics

        batch_size = (
            input_ids.size(0)
        )

        total_loss += (
            loss.item()
            *
            batch_size
        )

        total_category_loss += (
            category_loss.item()
            *
            batch_size
        )

        total_sentiment_loss += (
            sentiment_loss.item()
            *
            batch_size
        )

        total_examples += batch_size

        # TensorBoard

        writer.add_scalar(
            "train/step_loss",
            loss.item(),
            global_step
        )

        writer.add_scalar(
            "train/step_category_loss",
            category_loss.item(),
            global_step
        )

        writer.add_scalar(
            "train/step_sentiment_loss",
            sentiment_loss.item(),
            global_step
        )

        global_step += 1

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return {
        "loss":
            total_loss
            /
            total_examples,

        "category_loss":
            total_category_loss
            /
            total_examples,

        "sentiment_loss":
            total_sentiment_loss
            /
            total_examples,

        "global_step":
            global_step,
    }

In [17]:

# ## Validation

@torch.no_grad()
def run_validation(
    model,
    loader
):

    model.eval()

    total_loss = 0.0
    total_category_loss = 0.0
    total_sentiment_loss = 0.0

    total_examples = 0

    category_probs = []
    category_targets = []

    sentiment_logits = []
    sentiment_targets = []


    for batch in loader:

        input_ids = batch[
            "input_ids"
        ].to(DEVICE)

        attention_mask = batch[
            "attention_mask"
        ].to(DEVICE)

        category_target = batch[
            "category_presence"
        ].to(DEVICE)

        sentiment_target = batch[
            "sentiment_labels"
        ].to(DEVICE)


        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )


        (
            loss,
            category_loss,
            sentiment_loss
        ) = calculate_loss(
            outputs,
            category_target,
            sentiment_target
        )


        batch_size = (
            input_ids.size(0)
        )

        total_loss += (
            loss.item()
            *
            batch_size
        )

        total_category_loss += (
            category_loss.item()
            *
            batch_size
        )

        total_sentiment_loss += (
            sentiment_loss.item()
            *
            batch_size
        )

        total_examples += batch_size


        category_probs.append(
            torch.sigmoid(
                outputs["category_logits"]
            ).cpu().numpy()
        )

        category_targets.append(
            category_target.cpu().numpy()
        )

        sentiment_logits.append(
            outputs["sentiment_logits"]
            .cpu()
            .numpy()
        )

        sentiment_targets.append(
            sentiment_target.cpu().numpy()
        )


    return {
        "loss":
            total_loss / total_examples,

        "category_loss":
            total_category_loss
            /
            total_examples,

        "sentiment_loss":
            total_sentiment_loss
            /
            total_examples,

        "category_probs":
            np.concatenate(
                category_probs,
                axis=0
            ),

        "category_targets":
            np.concatenate(
                category_targets,
                axis=0
            ),

        "sentiment_logits":
            np.concatenate(
                sentiment_logits,
                axis=0
            ),

        "sentiment_targets":
            np.concatenate(
                sentiment_targets,
                axis=0
            ),
    }




In [18]:
# Tune category thresholds


def find_best_threshold(
    probabilities,
    targets
):

    best_threshold = 0.5
    best_f1 = -1.0

    for threshold in np.arange(
        0.05,#0.10,
        0.96,#0.91,
        0.01
    ):

        predictions = (
            probabilities
            >=
            threshold
        ).astype(int)

        score = f1_score(
            targets,
            predictions,
            zero_division=0
        )

        if score > best_f1:

            best_f1 = score
            best_threshold = (
                float(threshold)
            )

    return (
        best_threshold,
        best_f1
    )


def tune_category_thresholds(results):

    thresholds = np.zeros(
        NUM_CATEGORIES,
        dtype=np.float32
    )

    for category_id in range(
        NUM_CATEGORIES
    ):

        threshold, score = (
            find_best_threshold(
                results["category_probs"][:,category_id],
                results["category_targets"][:,category_id]
            )
        )

        thresholds[
            category_id
        ] = threshold

    return thresholds



In [19]:
def calculate_category_metrics(
    category_probs,
    category_targets,
    thresholds
):

    predictions = (
        category_probs
        >=
        np.asarray(thresholds)
    )

    targets = (
        category_targets.astype(int)
    )

    accuracy = accuracy_score(
        targets,
        predictions
    )

    macro_precision = precision_score(
        targets,
        predictions,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        targets,
        predictions,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        targets,
        predictions,
        average="macro",
        zero_division=0
    )

    micro_f1 = f1_score(
        targets,
        predictions,
        average="micro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        targets,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {

        "accuracy":
            accuracy,

        "macro_precision":
            macro_precision,

        "macro_recall":
            macro_recall,

        "macro_f1":
            macro_f1,

        "micro_f1":
            micro_f1,

        "weighted_f1":
            weighted_f1
    }


In [ ]:

writer = SummaryWriter(
    config["experiment_name"]
)

best_val_macro_f1 = -float("inf")

best_state = None
best_epoch = 0

best_thresholds = None

epochs_without_improvement = 0

history = []

global_step = 0


for epoch in range(
    1,
    config["num_epochs"] + 1
):

    print()
    print("=" * 60)

    print(
        f"Epoch "
        f"{epoch}/"
        f"{config['num_epochs']}"
    )

    print("=" * 60)


    start_time = (
        time.perf_counter()
    )


    # Training

    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        writer,
        global_step
    )

    global_step = (
        train_metrics["global_step"]
    )


    # Validation

    val_metrics = run_validation(
        model,
        val_loader
    )


    # Calculate category metrics using fixed thresholds

    fixed_thresholds = np.full(
      NUM_CATEGORIES,
      0.5,
      dtype=np.float32
    )

    category_metrics = calculate_category_metrics(
        val_metrics["category_probs"],
        val_metrics["category_targets"],
        fixed_thresholds
    )

    epoch_thresholds = ( #saved in checkpoint
      tune_category_thresholds(
          val_metrics
      )
    )


    epoch_time = (
        time.perf_counter()
        -
        start_time
    )


    # Print losses

    print(
        f"\nTrain loss: "
        f"{train_metrics['loss']:.4f}"
    )

    print(
        f"  category: "
        f"{train_metrics['category_loss']:.4f}"
    )

    print(
        f"  sentiment: "
        f"{train_metrics['sentiment_loss']:.4f}"
    )


    print(
        f"\nVal loss: "
        f"{val_metrics['loss']:.4f}"
    )

    print(
        f"  category: "
        f"{val_metrics['category_loss']:.4f}"
    )

    print(
        f"  sentiment: "
        f"{val_metrics['sentiment_loss']:.4f}"
    )


    # Print validation metrics

    print(
        f"\nValidation category metrics "
        f"(tuned thresholds):"
    )

    print(
        f"  Accuracy:    "
        f"{category_metrics['accuracy']:.4f}"
    )

    print(
        f"  Macro-F1:    "
        f"{category_metrics['macro_f1']:.4f}"
    )

    print(
        f"  Micro-F1:    "
        f"{category_metrics['micro_f1']:.4f}"
    )

    print(
        f"  Weighted-F1: "
        f"{category_metrics['weighted_f1']:.4f}"
    )


    print(
        f"\nEpoch time: "
        f"{epoch_time:.2f}s"
    )


    # TensorBoard

    writer.add_scalar(
        "train/loss",
        train_metrics["loss"],
        epoch
    )

    writer.add_scalar(
        "train/category_loss",
        train_metrics["category_loss"],
        epoch
    )

    writer.add_scalar(
        "train/sentiment_loss",
        train_metrics["sentiment_loss"],
        epoch
    )


    writer.add_scalar(
        "validation/loss",
        val_metrics["loss"],
        epoch
    )

    writer.add_scalar(
        "validation/category_loss",
        val_metrics["category_loss"],
        epoch
    )

    writer.add_scalar(
        "validation/sentiment_loss",
        val_metrics["sentiment_loss"],
        epoch
    )


    writer.add_scalar(
        "validation/accuracy",
        category_metrics["accuracy"],
        epoch
    )

    writer.add_scalar(
        "validation/macro_f1",
        category_metrics["macro_f1"],
        epoch
    )

    writer.add_scalar(
        "validation/micro_f1",
        category_metrics["micro_f1"],
        epoch
    )

    writer.add_scalar(
        "validation/weighted_f1",
        category_metrics["weighted_f1"],
        epoch
    )


    writer.add_scalar(
        "training/epoch_time",
        epoch_time,
        epoch
    )

    writer.add_scalar(
        "training/learning_rate",
        optimizer.param_groups[0]["lr"],
        epoch
    )


    # Save checkpoint

    output_dir = Path(
        config["model_folder"]
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    checkpoint_path = (
        output_dir
        /
        f"checkpoint_epoch_{epoch}.pt"
    )


    torch.save(
        {
            "epoch":
                epoch,

            "model_state_dict":
                model.state_dict(),

            "categories":
                CATEGORIES,

            "sentiments":
                SENTIMENTS,

            "category_thresholds":
                epoch_thresholds,

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "train_loss":
                train_metrics["loss"],

            "val_loss":
                val_metrics["loss"],

            "val_accuracy":
                category_metrics["accuracy"],

            "val_macro_f1":
                category_metrics["macro_f1"],

            "val_micro_f1":
                category_metrics["micro_f1"],

            "val_weighted_f1":
                category_metrics["weighted_f1"],
        },
        checkpoint_path
    )


    print(
        f"Saved checkpoint: "
        f"{checkpoint_path}"
    )


    # History

    history.append({

        "epoch":
            epoch,

        "train_loss":
            train_metrics["loss"],

        "train_category_loss":
            train_metrics["category_loss"],

        "train_sentiment_loss":
            train_metrics["sentiment_loss"],

        "val_loss":
            val_metrics["loss"],

        "val_category_loss":
            val_metrics["category_loss"],

        "val_sentiment_loss":
            val_metrics["sentiment_loss"],

        "val_accuracy":
            category_metrics["accuracy"],

        "val_macro_f1":
            category_metrics["macro_f1"],

        "val_micro_f1":
            category_metrics["micro_f1"],

        "val_weighted_f1":
            category_metrics["weighted_f1"],

        "epoch_time":
            epoch_time
    })


    # Best model

    current_macro_f1 = (
        category_metrics["macro_f1"]
    )


    if (
        current_macro_f1
        >
        best_val_macro_f1
    ):

        best_val_macro_f1 = (
            current_macro_f1
        )

        best_state = copy.deepcopy(
            model.state_dict()
        )

        best_thresholds = (
            epoch_thresholds.copy()
        )

        best_epoch = epoch

        epochs_without_improvement = 0

        print(
            "\nNew best model!"
        )

        print(
            f"Validation macro-F1: "
            f"{current_macro_f1:.4f}"
        )

    else:

        epochs_without_improvement += 1

        print(
            f"\nNo improvement "
            f"({epochs_without_improvement}/"
            f"{config['early_stopping_patience']})"
        )


    # Early stopping

    if (
        epochs_without_improvement
        >=
        config["early_stopping_patience"]
    ):

        print(
            "\nEarly stopping."
        )

        break


# Save best model

best_model_path = (
    Path(config["model_folder"])
    /
    "best_model.pt"
)


torch.save(
    {
        "epoch":
            best_epoch,

        "model_state_dict":
            best_state,

        "categories":
            CATEGORIES,

        "sentiments":
            SENTIMENTS,

        "category_thresholds":
            best_thresholds,

        "best_val_macro_f1":
            best_val_macro_f1
    },
    best_model_path
)


print()
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print(
    f"Best epoch: "
    f"{best_epoch}"
)

print(
    f"Best validation macro-F1: "
    f"{best_val_macro_f1:.4f}"
)

print(
    f"Best thresholds: "
    f"{best_thresholds}"
)

print(
    f"Saved: "
    f"{best_model_path}"
)


writer.close()


Epoch 1/20


Training:   0%|          | 0/324 [00:00<?, ?it/s]

/tmp/ipykernel_1427/1154088953.py:100: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()



Train loss: 2.2729
  category: 1.1128
  sentiment: 1.1601

Val loss: 1.9795
  category: 1.0077
  sentiment: 0.9717

Validation category metrics (tuned thresholds):
  Accuracy:    0.0525
  Macro-F1:    0.3898
  Micro-F1:    0.4199
  Weighted-F1: 0.4862

Epoch time: 105.34s
Saved checkpoint: weights/checkpoint_epoch_1.pt

New best model!
Validation macro-F1: 0.3898

Epoch 2/20


Training:   0%|          | 0/324 [00:00<?, ?it/s]

...


Train loss: 0.4558
  category: 0.1357
  sentiment: 0.3201

Val loss: 1.4453
  category: 0.4081
  sentiment: 1.0372

Validation category metrics (tuned thresholds):
  Accuracy:    0.5382
  Macro-F1:    0.8215
  Micro-F1:    0.8457
  Weighted-F1: 0.8475

Epoch time: 103.36s
Saved checkpoint: weights/checkpoint_epoch_18.pt

New best model!
Validation macro-F1: 0.8215

Epoch 19/20


Training:   0%|          | 0/324 [00:00<?, ?it/s]


Train loss: 0.4443
  category: 0.1295
  sentiment: 0.3147

Val loss: 1.4166
  category: 0.4020
  sentiment: 1.0145

Validation category metrics (tuned thresholds):
  Accuracy:    0.5287
  Macro-F1:    0.8213
  Micro-F1:    0.8442
  Weighted-F1: 0.8462

Epoch time: 105.68s
Saved checkpoint: weights/checkpoint_epoch_19.pt

No improvement (1/10)

Epoch 20/20


Training:   0%|          | 0/324 [00:00<?, ?it/s]


Train loss: 0.4386
  category: 0.1252
  sentiment: 0.3133

Val loss: 1.4229
  category: 0.4027
  sentiment: 1.0201

Validation category metrics (tuned thresholds):
  Accuracy:    0.5287
  Macro-F1:    0.8208
  Micro-F1:    0.8441
  Weighted-F1: 0.8460

Epoch time: 105.26s
Saved checkpoint: weights/checkpoint_epoch_20.pt

No improvement (2/10)

TRAINING COMPLETE
Best epoch: 18
Best validation macro-F1: 0.8215
Best thresholds: [0.49 0.63 0.81 0.94 0.33 0.71 0.62 0.48 0.86 0.58 0.48]
Saved: weights/best_model.pt


In [ ]:
# Restore best model

if best_state is not None:

    model.load_state_dict(
        best_state
    )

print()
print(
    f"Best epoch: {best_epoch}"
)

print(
    f"Best validation macro-F1: "
    f"{best_val_macro_f1:.4f}"
)



# Final validation pass

val_results = run_validation(
    model,
    val_loader
)

print(
    f"\nValidation loss: "
    f"{val_results['loss']:.4f}"
)


# Tune thresholds for the selected model

category_thresholds = np.zeros(
    NUM_CATEGORIES,
    dtype=np.float32
)

print(
    "\nFinal validation thresholds:"
)

for category_id, category in (
    ID_TO_CATEGORY.items()
):

    threshold, score = find_best_threshold(
        val_results["category_probs"][
            :,
            category_id
        ],
        val_results["category_targets"][
            :,
            category_id
        ]
    )

    category_thresholds[
        category_id
    ] = threshold

    print(
        f"{category:15s} "
        f"threshold={threshold:.2f} "
        f"F1={score:.4f}"
    )


# Save final best model

output_dir = Path(
    config["model_folder"]
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

best_model_path = (
    output_dir
    /
    "best_model.pt"
)

torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "categories":
            CATEGORIES,

        "sentiments":
            SENTIMENTS,

        "category_thresholds":
            category_thresholds,

        "model_name":
            config["model_name"],

        "max_length":
            config["max_length"],

        "best_epoch":
            best_epoch,

        "best_val_macro_f1":
            best_val_macro_f1,

        "final_val_loss":
            val_results["loss"],
    },
    best_model_path
)


tokenizer.save_pretrained(
    output_dir
)

print(
    "\nSaved:",
    best_model_path
)


Best epoch: 18
Best validation macro-F1: 0.8215

Validation loss: 1.4453

Final validation thresholds:
Baterija        threshold=0.49 F1=0.9478
Kamera          threshold=0.63 F1=0.9459
Ekran           threshold=0.81 F1=0.8617
Memorija        threshold=0.94 F1=0.6667
Zvučnici        threshold=0.33 F1=0.9009
Izgled          threshold=0.71 F1=0.8571
Hardver         threshold=0.62 F1=0.8055
Softver         threshold=0.48 F1=0.8052
Cena            threshold=0.86 F1=0.8810
Performanse     threshold=0.58 F1=0.7782
Opšta ocena     threshold=0.48 F1=0.8598

Saved: weights/best_model.pt


In [ ]:
checkpoint = torch.load(
    "weights/checkpoint_epoch_20.pt",
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(DEVICE)
model.eval()

category_thresholds = checkpoint["category_thresholds"]

In [23]:
# Evaluate category detection

def evaluate_categories(
    results,
    thresholds
):

    probabilities = results[
        "category_probs"
    ]

    targets = results[
        "category_targets"
    ]

    predictions = (
        probabilities
        >=
        thresholds
    ).astype(int)


    # Per-category metrics

    metrics = []

    for category_id, category in (
        ID_TO_CATEGORY.items()
    ):

        y_true = targets[
            :,
            category_id
        ]

        y_pred = predictions[
            :,
            category_id
        ]

        metrics.append({
            "category":
                category,

            "accuracy":
                accuracy_score(
                    y_true,
                    y_pred
                ),

            "precision":
                precision_score(
                    y_true,
                    y_pred,
                    zero_division=0
                ),

            "recall":
                recall_score(
                    y_true,
                    y_pred,
                    zero_division=0
                ),

            "f1":
                f1_score(
                    y_true,
                    y_pred,
                    zero_division=0
                ),
            "support": int(targets[:, category_id].sum()),
        })


    # Flatten multilabel predictions

    y_true = targets.reshape(-1)
    y_pred = predictions.reshape(-1)


    # Overall metrics

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    macro_precision = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    micro_precision = precision_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    micro_recall = recall_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    micro_f1 = f1_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )


    return {
        "predictions":
            predictions,

        "per_category":
            metrics,

        "accuracy":
            accuracy,

        "macro_precision":
            macro_precision,

        "macro_recall":
            macro_recall,

        "macro_f1":
            macro_f1,

        "micro_precision":
            micro_precision,

        "micro_recall":
            micro_recall,

        "micro_f1":
            micro_f1,

        "weighted_f1":
            weighted_f1,
    }

val_results = run_validation(
    model,
    val_loader
)


val_category_metrics = evaluate_categories(
    val_results,
    category_thresholds
)

print(
    f"Category accuracy: "
    f"{val_category_metrics['accuracy']:.4f}"
)

print(
    f"Category macro-P: "
    f"{val_category_metrics['macro_precision']:.4f}"
)

print(
    f"Category macro-R: "
    f"{val_category_metrics['macro_recall']:.4f}"
)

print(
    f"Category macro-F1: "
    f"{val_category_metrics['macro_f1']:.4f}"
)

print(
    f"Category micro-P: "
    f"{val_category_metrics['micro_precision']:.4f}"
)

print(
    f"Category micro-R: "
    f"{val_category_metrics['micro_recall']:.4f}"
)

print(
    f"Category micro-F1: "
    f"{val_category_metrics['micro_f1']:.4f}"
)

print(
    f"Category weighted-F1: "
    f"{val_category_metrics['weighted_f1']:.4f}"
)

print()

for metric in val_category_metrics["per_category"]:

    print(
        f"{metric['category']:15s} "
        f"P={metric['precision']:.4f} "
        f"R={metric['recall']:.4f} "
        f"F1={metric['f1']:.4f} "
        f"support={metric['support']:.4f}"
    )

Category accuracy: 0.9409
Category macro-P: 0.9048
Category macro-R: 0.9216
Category macro-F1: 0.9128
Category micro-P: 0.9409
Category micro-R: 0.9409
Category micro-F1: 0.9409
Category weighted-F1: 0.9415

Baterija        P=0.9121 R=0.9864 F1=0.9478 support=221.0000
Kamera          P=0.9272 R=0.9655 F1=0.9459 support=145.0000
Ekran           P=0.8804 R=0.8438 F1=0.8617 support=96.0000
Memorija        P=0.6000 R=0.7500 F1=0.6667 support=16.0000
Zvučnici        P=0.8475 R=0.9615 F1=0.9009 support=52.0000
Izgled          P=0.8675 R=0.8471 F1=0.8571 support=85.0000
Hardver         P=0.7712 R=0.8429 F1=0.8055 support=140.0000
Softver         P=0.7789 R=0.8333 F1=0.8052 support=186.0000
Cena            P=0.8916 R=0.8706 F1=0.8810 support=85.0000
Performanse     P=0.7440 R=0.8158 F1=0.7782 support=114.0000
Opšta ocena     P=0.8338 R=0.8875 F1=0.8598 support=311.0000


In [ ]:
# Oracle-level Sentiment Evaluation
#
# Oracle:
#   Pretpostavlja da je kategorija detektovana
#   tačno, tj. koristi GOLD kategoriju.

def evaluate_sentiment(
    results
):

    category_targets = results[
        "category_targets"
    ]

    sentiment_logits = results[
        "sentiment_logits"
    ]

    sentiment_targets = results[
        "sentiment_targets"
    ]

    predicted_sentiments = np.argmax(
        sentiment_logits,
        axis=2
    )

    all_gold = []
    all_pred = []

    per_category = []


    # Per-category Oracle evaluation

    for category_id, category in (
        ID_TO_CATEGORY.items()
    ):

        valid = (
            category_targets[
                :,
                category_id
            ]
            == 1
        ) & (
            sentiment_targets[
                :,
                category_id
            ]
            >= 0
        )

        if not valid.any():
            continue


        gold = sentiment_targets[
            valid,
            category_id
        ]

        predicted = predicted_sentiments[
            valid,
            category_id
        ]


        # Collect for overall metrics

        all_gold.extend(
            gold.tolist()
        )

        all_pred.extend(
            predicted.tolist()
        )


        # Per-category metrics

        accuracy = accuracy_score(
            gold,
            predicted
        )

        precision = precision_score(
            gold,
            predicted,
            average="macro",
            labels=np.arange(
                NUM_SENTIMENTS
            ),
            zero_division=0
        )

        recall = recall_score(
            gold,
            predicted,
            average="macro",
            labels=np.arange(
                NUM_SENTIMENTS
            ),
            zero_division=0
        )

        macro_f1 = f1_score(
            gold,
            predicted,
            average="macro",
            labels=np.arange(
                NUM_SENTIMENTS
            ),
            zero_division=0
        )

        weighted_f1 = f1_score(
            gold,
            predicted,
            average="weighted",
            labels=np.arange(
                NUM_SENTIMENTS
            ),
            zero_division=0
        )


        per_category.append({

            "category":
                category,

            "accuracy":
                accuracy,

            "precision":
                precision,

            "recall":
                recall,

            "macro_f1":
                macro_f1,

            "weighted_f1":
                weighted_f1,

            "support":
                len(gold),
        })


    # Overall Oracle metrics

    accuracy = accuracy_score(
        all_gold,
        all_pred
    )

    macro_precision = precision_score(
        all_gold,
        all_pred,
        average="macro",
        labels=np.arange(
            NUM_SENTIMENTS
        ),
        zero_division=0
    )

    macro_recall = recall_score(
        all_gold,
        all_pred,
        average="macro",
        labels=np.arange(
            NUM_SENTIMENTS
        ),
        zero_division=0
    )

    macro_f1 = f1_score(
        all_gold,
        all_pred,
        average="macro",
        labels=np.arange(
            NUM_SENTIMENTS
        ),
        zero_division=0
    )

    micro_precision = precision_score(
        all_gold,
        all_pred,
        average="micro",
        labels=np.arange(
            NUM_SENTIMENTS
        ),
        zero_division=0
    )

    micro_recall = recall_score(
        all_gold,
        all_pred,
        average="micro",
        labels=np.arange(
            NUM_SENTIMENTS
        ),
        zero_division=0
    )

    micro_f1 = f1_score(
        all_gold,
        all_pred,
        average="micro",
        labels=np.arange(
            NUM_SENTIMENTS
        ),
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_gold,
        all_pred,
        average="weighted",
        labels=np.arange(
            NUM_SENTIMENTS
        ),
        zero_division=0
    )


    return {

        "accuracy":
            accuracy,

        "macro_precision":
            macro_precision,

        "macro_recall":
            macro_recall,

        "macro_f1":
            macro_f1,

        "micro_precision":
            micro_precision,

        "micro_recall":
            micro_recall,

        "micro_f1":
            micro_f1,

        "weighted_f1":
            weighted_f1,

        "per_category":
            per_category,
    }



In [ ]:
# Flat-level End-to-End evaluacija

def evaluate_flat_e2e(
    results,
    thresholds
):

    category_probs = results["category_probs"]
    category_targets = results["category_targets"]
    sentiment_logits = results["sentiment_logits"]
    sentiment_targets = results["sentiment_targets"]

    predicted_sentiments = np.argmax(
        sentiment_logits,
        axis=2
    )

    # Only the 4 actual sentiment classes.
    active_labels = [0, 1, 2, 3]

    gold_flat = []
    pred_flat = []

    for category_id in range(NUM_CATEGORIES):

        gold_category = category_targets[:, category_id]
        gold_sentiment = sentiment_targets[:, category_id]


        predicted_category = (
            category_probs[:, category_id]
            >
            thresholds[category_id]
        )

        predicted_sentiment = predicted_sentiments[:, category_id]

        # Gold label
        #
        # Existing category; sentiment
        # Missing category; -1

        gold_labels = np.full(
            len(gold_category),
            -1,
            dtype=np.int64
        )

        valid_gold = (
            (gold_category == 1)
            &
            (gold_sentiment >= 0)
        )

        gold_labels[valid_gold] = (
            gold_sentiment[valid_gold]
        )

        # Predicted label
        #
        # ACD says present; predicted sentiment
        # ACD says absent; -1

        pred_labels = np.full(
            len(predicted_category),
            -1,
            dtype=np.int64
        )

        pred_labels[predicted_category] = (
            predicted_sentiment[predicted_category]
        )

        gold_flat.extend(
            gold_labels.tolist()
        )

        pred_flat.extend(
            pred_labels.tolist()
        )


    # Flat-level metrics
    #
    # evaluate only the 4 sentiment classes.


    accuracy = accuracy_score(
        gold_flat,
        pred_flat
    )

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            gold_flat,
            pred_flat,
            labels=active_labels,
            average="macro",
            zero_division=0
        )
    )

    micro_precision, micro_recall, micro_f1, _ = (
        precision_recall_fscore_support(
            gold_flat,
            pred_flat,
            labels=active_labels,
            average="micro",
            zero_division=0
        )
    )

    weighted_f1 = f1_score(
        gold_flat,
        pred_flat,
        labels=active_labels,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy" : accuracy,
        "macro_f1": macro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,

        "micro_f1": micro_f1,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,

        "weighted_f1": weighted_f1,
    }

In [ ]:
# Tuple-level End-to-End evaluation

def evaluate_tuple_e2e(
    results,
    thresholds
):

    category_probs = results["category_probs"]
    category_targets = results["category_targets"]
    sentiment_logits = results["sentiment_logits"]
    sentiment_targets = results["sentiment_targets"]

    predicted_sentiments = np.argmax(
        sentiment_logits,
        axis=2
    )

    total_tp = 0
    total_fp = 0
    total_fn = 0

    document_f1_scores = []

    num_documents = category_targets.shape[0]

    for document_id in range(num_documents):

        gold_tuples = set()
        predicted_tuples = set()

        # Gold tuples

        for category_id in range(NUM_CATEGORIES):

            if (
                category_targets[
                    document_id,
                    category_id
                ] == 1
                and
                sentiment_targets[
                    document_id,
                    category_id
                ] >= 0
            ):

                gold_tuples.add(
                    (
                        category_id,
                        int(
                            sentiment_targets[
                                document_id,
                                category_id
                            ]
                        )
                    )
                )

        # Predicted tuples

        for category_id in range(NUM_CATEGORIES):

            emitted = (
                category_probs[
                    document_id,
                    category_id
                ]
                >
                thresholds[category_id]
            )

            if emitted:

                predicted_tuples.add(
                    (
                        category_id,
                        int(
                            predicted_sentiments[
                                document_id,
                                category_id
                            ]
                        )
                    )
                )

        # Compare tuple sets

        tp = len(
            gold_tuples &
            predicted_tuples
        )

        fp = len(
            predicted_tuples -
            gold_tuples
        )

        fn = len(
            gold_tuples -
            predicted_tuples
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn

        # Document-level F1
        #
        # 0.0 when there are no TP/FP/FN

        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )

        document_f1 = (
            (2 * precision * recall) /
            (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

        document_f1_scores.append(
            document_f1
        )

    # Tuple micro-F1

    micro_precision = (
        total_tp /
        (total_tp + total_fp)
        if (total_tp + total_fp) > 0
        else 0.0
    )

    micro_recall = (
        total_tp /
        (total_tp + total_fn)
        if (total_tp + total_fn) > 0
        else 0.0
    )

    micro_f1 = (
        2 * micro_precision * micro_recall /
        (micro_precision + micro_recall)
        if (micro_precision + micro_recall) > 0
        else 0.0
    )

    # Tuple macro-F1

    macro_f1 = np.mean(
        document_f1_scores
    )

    return {
        "micro_precision":
            micro_precision,

        "micro_recall":
            micro_recall,

        "micro_f1":
            micro_f1,

        "macro_f1":
            macro_f1,

        "tp":
            total_tp,

        "fp":
            total_fp,

        "fn":
            total_fn,
    }

In [ ]:
# Save thresholds into best-model checkpoint

print(best_model_path)

checkpoint = torch.load(
    best_model_path,
    map_location=DEVICE,
    weights_only=False
)

checkpoint[
    "category_thresholds"
] = category_thresholds

torch.save(
    checkpoint,
    best_model_path
)

print(
    "Updated checkpoint with thresholds."
)

weights/best_model.pt
Updated checkpoint with thresholds.


In [27]:
# ## Final test evaluation
#
# Test se evaluira nakon:
#
#   - selekcije najboljeg modela
#   - izbora najboljih pragova(threshold-ova)

def run_test(
    model,
    test_loader,
    category_loss_fn,
    sentiment_loss_fn,
    thresholds
):

    results = run_validation(
        model,
        test_loader
    )


    # Category metrics

    category_metrics = (
        evaluate_categories(
            results,
            thresholds
        )
    )

    oracle_metrics = (
        evaluate_sentiment(
            results
        )
    )


    flat_e2e_metrics = (
        evaluate_flat_e2e(
            results,
            thresholds
        )
    )

    tuple_e2e_metrics = (
        evaluate_tuple_e2e(
            results,
            thresholds
        )
    )



    # Print

    print()
    print("=" * 60)
    print("TEST RESULTS")
    print("=" * 60)


    # Loss

    print(
        f"Loss: "
        f"{results['loss']:.4f}"
    )

    print(
        f"Category loss: "
        f"{results['category_loss']:.4f}"
    )

    print(
        f"Sentiment loss: "
        f"{results['sentiment_loss']:.4f}"
    )


    # Category detection

    print()
    print("CATEGORY DETECTION")
    print("-" * 60)

    print(
        f"Accuracy:       "
        f"{category_metrics['accuracy']:.4f}"
    )

    print(
        f"Macro precision: "
        f"{category_metrics['macro_precision']:.4f}"
    )

    print(
        f"Macro recall:    "
        f"{category_metrics['macro_recall']:.4f}"
    )

    print(
        f"Macro F1:        "
        f"{category_metrics['macro_f1']:.4f}"
    )

    print(
        f"Micro precision: "
        f"{category_metrics['micro_precision']:.4f}"
    )

    print(
        f"Micro recall:    "
        f"{category_metrics['micro_recall']:.4f}"
    )

    print(
        f"Micro F1:        "
        f"{category_metrics['micro_f1']:.4f}"
    )

    print(
        f"Weighted F1:     "
        f"{category_metrics['weighted_f1']:.4f}"
    )


    # Oracle Sentiment

    print()
    print("ORACLE SENTIMENT")
    print("-" * 60)

    print(
        f"Accuracy:       "
        f"{oracle_metrics['accuracy']:.4f}"
    )

    print(
        f"Macro precision: "
        f"{oracle_metrics['macro_precision']:.4f}"
    )

    print(
        f"Macro recall:    "
        f"{oracle_metrics['macro_recall']:.4f}"
    )

    print(
        f"Macro F1:        "
        f"{oracle_metrics['macro_f1']:.4f}"
    )

    print(
        f"Micro precision: "
        f"{oracle_metrics['micro_precision']:.4f}"
    )

    print(
        f"Micro recall:    "
        f"{oracle_metrics['micro_recall']:.4f}"
    )

    print(
        f"Micro F1:        "
        f"{oracle_metrics['micro_f1']:.4f}"
    )

    print(
        f"Weighted F1:     "
        f"{oracle_metrics['weighted_f1']:.4f}"
    )





    # Flat-level End-to-End

    print()
    print("FLAT-LEVEL END-TO-END")
    print("-" * 60)

    print(
        f"Accuracy:       "
        f"{flat_e2e_metrics['accuracy']:.4f}"
    )

    print(
        f"Macro precision: "
        f"{flat_e2e_metrics['macro_precision']:.4f}"
    )

    print(
        f"Macro recall:    "
        f"{flat_e2e_metrics['macro_recall']:.4f}"
    )

    print(
        f"Macro F1:        "
        f"{flat_e2e_metrics['macro_f1']:.4f}"
    )

    print(
        f"Micro precision: "
        f"{flat_e2e_metrics['micro_precision']:.4f}"
    )

    print(
        f"Micro recall:    "
        f"{flat_e2e_metrics['micro_recall']:.4f}"
    )

    print(
        f"Micro F1:        "
        f"{flat_e2e_metrics['micro_f1']:.4f}"
    )

    print(
        f"Weighted F1:     "
        f"{flat_e2e_metrics['weighted_f1']:.4f}"
    )


    # Tuple-level End-to-End

    print()
    print("TUPLE-LEVEL END-TO-END")
    print("-" * 60)

    print(
        f"Tuple Micro-F1: "
        f"{tuple_e2e_metrics['micro_f1']:.4f}"
    )

    print(
        f"Tuple Macro-F1: "
        f"{tuple_e2e_metrics['macro_f1']:.4f}"
    )

    print(
        f"TP:             "
        f"{tuple_e2e_metrics['tp']}"
    )

    print(
        f"FP:             "
        f"{tuple_e2e_metrics['fp']}"
    )

    print(
        f"FN:             "
        f"{tuple_e2e_metrics['fn']}"
    )


    # Per-category category detection

    print()

    for metric in category_metrics["per_category"]:
        print(
            f"{metric['category']:15s} "
            f"P={metric['precision']:.4f} "
            f"R={metric['recall']:.4f} "
            f"F1={metric['f1']:.4f} "
            f"support={metric['support']:.4f}"
        )


    # Return

    return {
        "results":
            results,

        "category_metrics":
            category_metrics,

        "flat_e2e_metrics":
            flat_e2e_metrics,

        "tuple_e2e_metrics":
            tuple_e2e_metrics,
    }


test_results = run_test(
    model,
    test_loader,
    category_loss_fn,
    sentiment_loss_fn,
    category_thresholds
)


TEST RESULTS
Loss: 1.4078
Category loss: 0.3765
Sentiment loss: 1.0313

CATEGORY DETECTION
------------------------------------------------------------
Accuracy:       0.9342
Macro precision: 0.8915
Macro recall:    0.9155
Macro F1:        0.9027
Micro precision: 0.9342
Micro recall:    0.9342
Micro F1:        0.9342
Weighted F1:     0.9351

ORACLE SENTIMENT
------------------------------------------------------------
Accuracy:       0.7702
Macro precision: 0.5043
Macro recall:    0.4862
Macro F1:        0.4933
Micro precision: 0.7702
Micro recall:    0.7702
Micro F1:        0.7702
Weighted F1:     0.7650

FLAT-LEVEL END-TO-END
------------------------------------------------------------
Accuracy:       0.8941
Macro precision: 0.4198
Macro recall:    0.4321
Macro F1:        0.4230
Micro precision: 0.6357
Micro recall:    0.6901
Micro F1:        0.6618
Weighted F1:     0.6551

TUPLE-LEVEL END-TO-END
------------------------------------------------------------
Tuple Micro-F1: 0.6618
Tup

In [ ]:
@torch.no_grad()
def predict_comment(
    text,
    model,
    tokenizer,
    thresholds
):

    model.eval()

    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=config["max_length"],
    )

    encoding = {
        key: value.to(DEVICE)
        for key, value in encoding.items()
    }

    outputs = model(
        input_ids=encoding["input_ids"],
        attention_mask=encoding["attention_mask"],
    )

    category_probs = torch.sigmoid(
        outputs["category_logits"]
    )[0].cpu().numpy()

    sentiment_probs = torch.softmax(
        outputs["sentiment_logits"],
        dim=-1
    )[0].cpu().numpy()

    predictions = []

    for category_id, category in ID_TO_CATEGORY.items():

        probability = float(
            category_probs[category_id]
        )

        threshold = float(
            thresholds[category_id]
        )

        present = probability >= threshold

        result = {
            "category": category,
            "probability": probability,
            "threshold": threshold,
            "present": present,
            "sentiment": None,
            "sentiment_probabilities": None,
        }

        if present:

            sentiment_id = int(
                np.argmax(
                    sentiment_probs[category_id]
                )
            )

            result["sentiment"] = (
                ID_TO_SENTIMENT[sentiment_id]
            )

            result["sentiment_probabilities"] = {
                ID_TO_SENTIMENT[i]: float(
                    sentiment_probs[category_id, i]
                )
                for i in range(NUM_SENTIMENTS)
            }

        predictions.append(result)

    for result in predictions:

      if not result["present"]:
          continue

      print(
          f"{result['category']:15s} "
          f"prob={result['probability']:.3f} "
          f"threshold={result['threshold']:.2f} "
          f"-> {result['sentiment']}"
      )

    return predictions

In [ ]:
text = "Pozdrav za sve!Baterija mi mnogo slabija drzi nakon poslednjeg abdejta 14.2,bukvalno se duplo brze isprazni nego ranije,a koristim sve iste aplikacije preko mikrog i aurore.Jel ima još neko ovakav problem,kakva su iskustva?"

predictions = predict_comment(
    text,
    model,
    tokenizer,
    category_thresholds,
)

#print(predictions)

Baterija        prob=0.989 threshold=0.49 -> Negativan
Softver         prob=0.878 threshold=0.48 -> Negativan


In [ ]:

for result in predictions:

    status = (
        "PRESENT"
        if result["present"]
        else "absent"
    )

    print(
        f"{result['category']:15s} "
        f"{result['probability']:.3f} "
        f"({result['threshold']:.2f}) "
        f"{status}"
    )

    if result["sentiment"]:

        print(
            f"    sentiment: "
            f"{result['sentiment']}"
        )

Baterija        0.989 (0.49) PRESENT
    sentiment: Negativan
Kamera          0.012 (0.63) absent
Ekran           0.005 (0.81) absent
Memorija        0.007 (0.94) absent
Zvučnici        0.005 (0.33) absent
Izgled          0.005 (0.71) absent
Hardver         0.020 (0.62) absent
Softver         0.878 (0.48) PRESENT
    sentiment: Negativan
Cena            0.005 (0.86) absent
Performanse     0.013 (0.58) absent
Opšta ocena     0.026 (0.48) absent


In [ ]:
from google.colab import files


files.download('weights/best_model.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
import shutil


source_path = '/content/weights/best_model.pt'
destination_path = '/content/drive/MyDrive/models/checkpoint/best_model_epoch_20_roberta_20_singlelr.pt'

#shutil.copy(source_path, destination_path)
shutil.copy(destination_path, source_path)

print("File saved to Google Drive successfully!")

File saved to Google Drive successfully!


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
checkpoint = torch.load(
    "weights/best_model.pt",
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(DEVICE)
model.eval()

category_thresholds = checkpoint["category_thresholds"]

In [ ]:

# Save final best model

output_dir = Path(
    config["model_folder"]
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

best_model_path = (
    output_dir
    /
    "best_model.pt"
)

torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "categories":
            CATEGORIES,

        "sentiments":
            SENTIMENTS,

        "category_thresholds":
            category_thresholds,

        "model_name":
            config["model_name"],

        "max_length":
            config["max_length"],

        "best_epoch":
            best_epoch,

        "best_val_macro_f1":
            best_val_macro_f1,

        "final_val_loss":
            val_results["loss"],
    },
    best_model_path
)


tokenizer.save_pretrained(
    output_dir
)

print(
    "\nSaved:",
    best_model_path
)


Saved: weights/best_model.pt
